# 01 — AOI and max-extent mask (exploration)

Visual check of the Phase 1 outputs from `python -m thwake baseline --step extent`
(method: `.ai/docs/03-methodology.md` §1.1–1.2). **Exploration only:** all logic lives in
`src/thwake/baseline.py`; this notebook only displays the files in `data/baseline/`.

What to check on the map:
- the max extent (blue) fills the upstream valleys and **ends at the dam wall** (magenta);
- nothing below the wall is included (white point = downstream check point);
- the AOI (orange) is a thin rim above the max extent; closed rim passes are red points;
- compare with the 2026 Sentinel-2 composite: the wall, spillway and cofferdam pond.

In [ ]:
import json

import ee
import geemap
import geopandas as gpd
import pandas as pd

from thwake import ee_auth
from thwake.baseline import extent_settings
from thwake.config import REPO_ROOT, load_config

cfg = load_config()
ee_auth.initialize(cfg)
s = extent_settings(cfg)

max_extent = gpd.read_file(s.max_extent_path)
aoi = gpd.read_file(s.aoi_path)
aoi_props = json.loads(s.aoi_path.read_text())["features"][0]["properties"]

In [ ]:
# Areas and metadata
m, a = max_extent.iloc[0], aoi.iloc[0]
pd.DataFrame(
    {
        "level_m_asl": [m.level_m_asl, a.level_m_asl],
        "area_km2": [m.area_km2, a.area_km2],
        "official_area_km2": [m.official_area_km2, None],
    },
    index=["max extent", "AOI"],
)

In [ ]:
print("FSL fallback:", m.fsl_is_fallback, "-", m.fsl_note)
print("DEM:", m.dem, "acquired", m.dem_acquired)
print("Wall axis:", m.wall_axis_source)
print("Pass closures:", aoi_props["pass_closures"])

In [ ]:
center = s.wall_axis[0]
Map = geemap.Map(center=(center[1], center[0]), zoom=12)

region = ee.Geometry.Rectangle(list(aoi.total_bounds)).buffer(2000)
cs = ee.ImageCollection(cfg.ee_collections["cloud_score_plus"])
s2 = (
    ee.ImageCollection(cfg.ee_collections["sentinel2_sr"])
    .filterBounds(region)
    .filterDate("2026-06-01", "2026-10-01")  # dry season, dam ~95% built
    .linkCollection(cs, ["cs"])
    .map(lambda i: i.updateMask(i.select("cs").gte(0.6)))
    .median()
)
dem = ee.ImageCollection(cfg.ee_collections["copernicus_dem_glo30"]).filterBounds(region).mosaic()

Map.addLayer(ee.Terrain.hillshade(dem.select("DEM")), {}, "Hillshade (GLO-30)", False)
Map.addLayer(
    dem.select("DEM"), {"min": 840, "max": 960, "palette": ["0000ff", "ffffff", "8b4513"]},
    "DEM 840–960 m", False,
)
Map.addLayer(s2, {"bands": ["B4", "B3", "B2"], "min": 200, "max": 2500, "gamma": 1.2}, "S2 Jun–Sep 2026")
Map.addLayer(geemap.gdf_to_ee(aoi[["geometry"]]).style(color="ffaa00", fillColor="ffaa0030"), {}, "AOI")
Map.addLayer(
    geemap.gdf_to_ee(max_extent[["geometry"]]).style(color="00ffff", fillColor="0060ff60"),
    {}, "Max extent",
)
Map.addLayer(ee.Geometry.LineString([list(p) for p in s.wall_axis]), {"color": "ff00ff"}, "Wall axis")
Map.addLayer(ee.Geometry.Point(list(s.seed)), {"color": "00ff00"}, "Reservoir seed (confluence)")
Map.addLayer(ee.Geometry.Point(list(s.downstream_point)), {"color": "ffffff"}, "Downstream check point")
if aoi_props["pass_closures"]:
    passes = ee.FeatureCollection(
        [ee.Geometry.Point([p["lon"], p["lat"]]) for p in aoi_props["pass_closures"]]
    )
    Map.addLayer(passes, {"color": "ff0000"}, "Closed rim passes")
Map

## Rim overflow passes (exploration)

The pre-dam DEM rim is low: with only the wall as a barrier, the basin overflows first at
≈913 m on the Makueni side, then ≈917.6 m at the Kitui abutment and ≈920.3 m elsewhere
(found in prompt 03; see `.ai/docs/08-open-questions.md` #17). Toggle the layer below to see
ground between FSL and the 920.5 m maximum flood level around the dam.

In [ ]:
fsl, mfl = s.fsl_m, cfg.settings["dam"]["maximum_flood_level_m_asl"]
band = dem.select("DEM").gt(fsl).And(dem.select("DEM").lte(mfl)).selfMask()
Map.addLayer(band, {"palette": ["ff0000"]}, f"Ground {fsl:g}–{mfl:g} m")
Map